# Type-Aware Compatibility Embedding — Full Pipeline

Everything in one place: dataset download → preprocessing → training → ONNX export.

**What this trains:**
ResNet-18 fine-tuned with triplet loss on Polyvore outfit pairs. The embedding space is
explicitly trained so compatible items cluster together — unlike FashionCLIP which was
trained for text-image retrieval, not compatibility.

**Settings before running:**
- Kaggle → Settings → Accelerator → **T4 GPU**
- Kaggle → Settings → Internet → **On**

**Expected runtime:** ~60-90 min (10 min download + 5 min setup + 45-60 min training)

**Output:** `type_aware_compatibility.zip` → extract and place both `.onnx` files in `backend/model/`

In [ ]:
# ── 1. Install ────────────────────────────────────────────────────────────────
!pip install -q torch torchvision onnx onnxruntime onnxscript scikit-learn tqdm Pillow huggingface_hub requests

In [ ]:
# ── 2. Imports + Config ───────────────────────────────────────────────────────
import os, json, pickle, random, time
from pathlib import Path
from itertools import combinations
from collections import defaultdict
import concurrent.futures
import requests

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from tqdm import tqdm
from huggingface_hub import snapshot_download

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ── Config ────────────────────────────────────────────────────────────────────
HF_REPO     = 'mvasil/polyvore-outfits'
SPLIT       = 'nondisjoint'   # nondisjoint has more training data than disjoint
N_OUTFITS   = 20_000          # outfits to train on (full dataset = ~68K)
VAL_RATIO   = 0.10            # 10% held out for validation
DIM_EMBED   = 64
IMG_SIZE    = 224
BATCH_SIZE  = 128             # reduce to 64 if OOM
LR          = 5e-5
MAX_EPOCHS  = 20
PATIENCE    = 5
MARGIN      = 0.3
DL_WORKERS  = 32              # parallel image download threads
DATA_DIR    = Path('/kaggle/working/polyvore')
OUTPUT_DIR  = Path('/kaggle/working')

print(f'Training on {N_OUTFITS:,} outfits  |  {SPLIT} split')

In [ ]:
# ── 3. Download metadata from HuggingFace (skipping the images directory) ─────
# Images are downloaded selectively in the next step — only what we need.
print('Downloading metadata files ...')
snapshot_download(
    repo_id=HF_REPO,
    repo_type='dataset',
    local_dir=str(DATA_DIR),
    ignore_patterns=['images/*', 'images/**'],
)

# Verify all required files landed
required = [
    f'{SPLIT}/train.json',
    f'{SPLIT}/typespaces.p',
    f'{SPLIT}/compatibility_test.txt',
    'polyvore_item_metadata.json',
]
for rel in required:
    assert (DATA_DIR / rel).exists(), f'Missing: {rel} — check dataset structure'
print('Metadata ready.')

# Show what was downloaded
for p in sorted(DATA_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(DATA_DIR)}  ({p.stat().st_size/1e3:.0f} KB)')

In [ ]:
# ── 4. Load metadata, typespaces, select outfits, collect needed image IDs ────

# Item metadata: item_id -> {semantic_category, ...}
print('Loading item metadata ...')
with open(DATA_DIR / 'polyvore_item_metadata.json') as f:
    meta = json.load(f)

im2cat = {
    str(iid): str(info.get('semantic_category', 'unk'))
    for iid, info in meta.items()
}
print(f'  Items in metadata: {len(im2cat):,}')

# Typespaces: list of (cat_a, cat_b) tuples → condition index
print('Loading typespaces ...')
with open(DATA_DIR / SPLIT / 'typespaces.p', 'rb') as f:
    try:
        ts_list = pickle.load(f)
    except UnicodeDecodeError:
        f.seek(0)
        ts_list = pickle.load(f, encoding='latin1')

typespaces   = {t: i for i, t in enumerate(ts_list)}
n_conditions = len(typespaces)
print(f'  Type-pair conditions: {n_conditions}')

# Select N_OUTFITS training outfits at random
print(f'Selecting {N_OUTFITS:,} outfits from train.json ...')
with open(DATA_DIR / SPLIT / 'train.json') as f:
    all_outfits = json.load(f)
random.shuffle(all_outfits)
selected = all_outfits[:N_OUTFITS]
print(f'  Selected {len(selected):,} / {len(all_outfits):,} outfits')

# Collect needed item IDs — training outfits + test outfits for AUC eval
needed_ids = set()
for outfit in selected:
    for item in outfit.get('items', []):
        needed_ids.add(str(item['item_id']))

with open(DATA_DIR / SPLIT / 'compatibility_test.txt') as f:
    for line in f:
        for iid in line.strip().split()[1:]:
            needed_ids.add(iid)

print(f'  Unique images needed: {len(needed_ids):,}')

In [ ]:
# ── 5. Download only the needed images in parallel ────────────────────────────
img_dir  = DATA_DIR / 'images'
img_dir.mkdir(exist_ok=True)
BASE_URL = f'https://huggingface.co/datasets/{HF_REPO}/resolve/main/images'

_sess = requests.Session()
_sess.headers.update({'User-Agent': 'Mozilla/5.0'})

def _download(item_id):
    dst = img_dir / f'{item_id}.jpg'
    if dst.exists() and dst.stat().st_size > 0:
        return True
    try:
        r = _sess.get(f'{BASE_URL}/{item_id}.jpg', timeout=30)
        if r.status_code == 200:
            dst.write_bytes(r.content)
            return True
    except Exception:
        pass
    return False

print(f'Downloading {len(needed_ids):,} images using {DL_WORKERS} threads ...')
t0  = time.time()
ids = list(needed_ids)
with concurrent.futures.ThreadPoolExecutor(max_workers=DL_WORKERS) as ex:
    results = list(tqdm(ex.map(_download, ids), total=len(ids), desc='images'))

ok       = sum(results)
elapsed  = time.time() - t0
missing  = len(ids) - ok
print(f'Downloaded: {ok:,}  |  Failed/missing: {missing:,}  |  Time: {elapsed:.0f}s')
if missing > len(ids) * 0.1:
    print(f'WARNING: more than 10% of images failed — check internet access and HF URL format')

In [ ]:
# ── 6. Dataset class + Model ──────────────────────────────────────────────────

class PolyvoreDataset(Dataset):
    """
    Returns (anchor, positive, negative, condition) triplets.

    Positive pairs: two items from the same outfit.
    Negative:       item from a different outfit but the same semantic category
                    as the positive (category-constrained hard negative).
    Condition:      index into the type-specific embedding space for this
                    (anchor_category, positive_category) pair.
    """
    def __init__(self, outfits, typespaces, im2cat, img_dir, transform):
        self.img_dir    = img_dir
        self.transform  = transform
        self.typespaces = typespaces

        cat2items = defaultdict(list)   # cat -> [(outfit_idx, item_id)]
        pos_pairs = []                  # (outfit_idx, a_id, p_id, cond, p_cat)

        for oidx, outfit in enumerate(outfits):
            items    = outfit.get('items', [])
            item_ids = [str(item['item_id']) for item in items]

            for iid in item_ids:
                cat = im2cat.get(iid, 'unk')
                cat2items[cat].append((oidx, iid))

            for i, j in combinations(range(len(item_ids)), 2):
                a_id  = item_ids[i]
                p_id  = item_ids[j]
                a_cat = im2cat.get(a_id, 'unk')
                p_cat = im2cat.get(p_id, 'unk')
                key   = (a_cat, p_cat)
                if key in typespaces:
                    pos_pairs.append((oidx, a_id, p_id, typespaces[key], p_cat))

        self.cat2items = cat2items
        self.pos_pairs = pos_pairs
        print(f'  {len(outfits):,} outfits → {len(pos_pairs):,} positive pairs')

    def _load(self, iid):
        p = self.img_dir / f'{iid}.jpg'
        try:
            return self.transform(Image.open(p).convert('RGB'))
        except Exception:
            return torch.zeros(3, IMG_SIZE, IMG_SIZE)

    def _neg(self, oidx, p_cat):
        cands = self.cat2items[p_cat]
        for _ in range(10):
            idx, nid = random.choice(cands)
            if idx != oidx:
                return nid
        return random.choice(cands)[1]

    def __len__(self):
        return len(self.pos_pairs)

    def __getitem__(self, idx):
        oidx, a_id, p_id, cond, p_cat = self.pos_pairs[idx]
        n_id = self._neg(oidx, p_cat)
        return (
            self._load(a_id),
            self._load(p_id),
            self._load(n_id),
            torch.tensor(cond, dtype=torch.long),
        )


class EmbeddingNet(nn.Module):
    """ResNet-18 backbone → 64-dim embedding."""
    def __init__(self, dim=64):
        super().__init__()
        r = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(r.children())[:-1])
        self.fc       = nn.Linear(512, dim)

    def forward(self, x):
        return self.fc(self.features(x).flatten(1))


class TypeAwareNet(nn.Module):
    """
    EmbeddingNet + learned type-specific diagonal masks.
    At training: applies the mask for the (anchor_cat, pos_cat) pair.
    At inference: returns the general L2-normalised embedding (no mask) —
                  this is still trained for compatibility via triplet loss.
    """
    def __init__(self, dim=64, n_cond=1):
        super().__init__()
        self.emb_net = EmbeddingNet(dim)
        arr = np.zeros([n_cond, dim], dtype=np.float32)
        ml  = max(1, dim // n_cond)
        for i in range(n_cond):
            s = (i * ml) % dim
            arr[i, s:s + ml] = 1.0
        self.masks = nn.Embedding(n_cond, dim)
        self.masks.weight = nn.Parameter(torch.from_numpy(arr))

    def forward(self, x, cond=None):
        e = self.emb_net(x)
        if cond is None:
            return F.normalize(e, p=2, dim=1)
        return F.normalize(e * F.relu(self.masks(cond)), p=2, dim=1)


model = TypeAwareNet(dim=DIM_EMBED, n_cond=n_conditions).to(DEVICE)
print(f'Parameters total:   {sum(p.numel() for p in model.parameters()):,}')
print(f'  EmbeddingNet:     {sum(p.numel() for p in model.emb_net.parameters()):,}')
print(f'  Type masks:       {sum(p.numel() for p in model.masks.parameters()):,}')

In [ ]:
# ── 7. Transforms, DataLoaders, Training ─────────────────────────────────────

T_train = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
T_eval = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

n_val   = int(len(selected) * VAL_RATIO)
outfits_train = selected[n_val:]
outfits_val   = selected[:n_val]

print('Building train dataset ...')
ds_train = PolyvoreDataset(outfits_train, typespaces, im2cat, img_dir, T_train)
print('Building val dataset ...')
ds_val   = PolyvoreDataset(outfits_val,   typespaces, im2cat, img_dir, T_eval)

train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
print(f'Train batches: {len(train_loader):,}  |  Val batches: {len(val_loader):,}')

# ── Training setup ────────────────────────────────────────────────────────────
criterion = nn.TripletMarginLoss(margin=MARGIN, p=2)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.5, min_lr=1e-6)

def run_epoch(loader, train):
    model.train() if train else model.eval()
    total, n = 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for a, p, neg, c in tqdm(loader, leave=False):
            a, p, neg, c = a.to(DEVICE), p.to(DEVICE), neg.to(DEVICE), c.to(DEVICE)
            loss = criterion(model(a, c), model(p, c), model(neg, c))
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total += loss.item(); n += 1
    return total / n

# ── Training loop ─────────────────────────────────────────────────────────────
best_loss, best_state, patience_cnt = float('inf'), None, 0
train_losses, val_losses = [], []

print('\nTraining ...')
for epoch in range(1, MAX_EPOCHS + 1):
    tl = run_epoch(train_loader, True)
    vl = run_epoch(val_loader,   False)
    scheduler.step(vl)
    train_losses.append(tl); val_losses.append(vl)
    print(f'Epoch {epoch:02d}/{MAX_EPOCHS}  '
          f'train={tl:.4f}  val={vl:.4f}  '
          f'lr={optimizer.param_groups[0]["lr"]:.2e}')
    if vl < best_loss:
        best_loss    = vl
        best_state   = {k: v.clone() for k, v in model.state_dict().items()}
        patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'Early stopping at epoch {epoch}  (best val loss: {best_loss:.4f})')
            break

model.load_state_dict(best_state)
print(f'\nBest val loss: {best_loss:.4f}')

In [ ]:
# ── 8. Training curves + Compatibility AUC ────────────────────────────────────

# Training curves
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_losses, label='Train')
ax.plot(val_losses,   label='Val')
ax.set(xlabel='Epoch', ylabel='Triplet loss', title='Training curves')
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=120)
plt.show()

# Compatibility AUC on test set
# Score = average pairwise cosine similarity of general embeddings across outfit items.
# Higher score = more compatible. Labels from compatibility_test.txt: 1=compatible, 0=not.
model.eval()
scores, labels = [], []

print('Evaluating compatibility AUC on test set ...')
with open(DATA_DIR / SPLIT / 'compatibility_test.txt') as f:
    lines = f.readlines()

for line in tqdm(lines):
    parts = line.strip().split()
    if len(parts) < 3:
        continue
    label, iids = int(parts[0]), parts[1:]
    embs = []
    for iid in iids:
        try:
            img    = Image.open(img_dir / f'{iid}.jpg').convert('RGB')
            tensor = T_eval(img).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                embs.append(model(tensor).cpu())
        except Exception:
            continue
    if len(embs) < 2:
        continue
    embs    = torch.cat(embs)                              # (N, dim)
    sim_mat = embs @ embs.T                                # cosine sim (already L2-normed)
    mask    = torch.triu(torch.ones_like(sim_mat), diagonal=1).bool()
    scores.append(sim_mat[mask].mean().item())
    labels.append(label)

auc = roc_auc_score(labels, scores)
print(f'\nCompatibility AUC: {auc:.4f}  (target >= 0.83)')
if auc >= 0.83:
    print('Target reached.')
elif auc >= 0.75:
    print('Acceptable — backend will still provide meaningful compatibility signals.')
else:
    print('Below target. Consider: training longer, full dataset, or increasing BATCH_SIZE.')

In [ ]:
# ── 9. Export to ONNX + INT8 ──────────────────────────────────────────────────
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType
import onnxruntime as ort

ONNX_PATH = str(OUTPUT_DIR / 'compatibility_embedding.onnx')
INT8_PATH = str(OUTPUT_DIR / 'compatibility_embedding_int8.onnx')


class InferenceWrapper(nn.Module):
    """Exports only the embedding backbone — no masks needed at inference.
    Input:  (B, 3, 224, 224) image
    Output: (B, 64) L2-normalised embedding
    """
    def __init__(self, net):
        super().__init__()
        self.net = net.emb_net

    def forward(self, x):
        return F.normalize(self.net(x), p=2, dim=1)


wrapper = InferenceWrapper(model).eval().cpu()
dummy   = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)

with torch.no_grad():
    out = wrapper(dummy)
print(f'Wrapper output: {tuple(out.shape)}  norm={out.norm().item():.4f}')

print(f'Exporting FP32 ONNX ...')
torch.onnx.export(
    wrapper, dummy, ONNX_PATH,
    input_names=['image'],
    output_names=['embedding'],
    dynamic_axes={'image': {0: 'batch'}, 'embedding': {0: 'batch'}},
    opset_version=17,
    do_constant_folding=True,
)
onnx.checker.check_model(ONNX_PATH)
print(f'  Saved: {ONNX_PATH}  ({os.path.getsize(ONNX_PATH)/1e6:.1f} MB)')

print('Quantising to INT8 ...')
quantize_dynamic(ONNX_PATH, INT8_PATH, weight_type=QuantType.QInt8)
print(f'  Saved: {INT8_PATH}  ({os.path.getsize(INT8_PATH)/1e6:.1f} MB)')

# Validate ONNX matches PyTorch
sess     = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
inp      = np.random.randn(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
pt_out   = wrapper(torch.from_numpy(inp)).detach().numpy()
onnx_out = sess.run(['embedding'], {'image': inp})[0]
diff     = float(np.abs(pt_out - onnx_out).max())
print(f'Max PyTorch vs ONNX diff: {diff:.6f}')
assert diff < 1e-3, f'Outputs differ too much: {diff}'
print(f'Embedding norm (should be ~1.0): {np.linalg.norm(onnx_out):.4f}')
print('ONNX export validated.')

In [ ]:
# ── 10. Package outputs ───────────────────────────────────────────────────────
import zipfile

zip_path = str(OUTPUT_DIR / 'type_aware_compatibility.zip')
to_zip   = [
    (ONNX_PATH,                                 'compatibility_embedding.onnx'),
    (INT8_PATH,                                  'compatibility_embedding_int8.onnx'),
    (str(OUTPUT_DIR / 'training_curves.png'),    'training_curves.png'),
]

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath, arcname in to_zip:
        if os.path.isfile(fpath):
            zf.write(fpath, arcname)
            print(f'  Added {arcname}  ({os.path.getsize(fpath)/1e6:.1f} MB)')

print(f'\nZip: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)')
print(f'Compatibility AUC achieved: {auc:.4f}')
print('\nNext steps:')
print('1. Kaggle sidebar → Data → Output → Save Version')
print('2. Download type_aware_compatibility.zip from the Output tab')
print('3. Extract and place in backend/model/:')
print('     compatibility_embedding_int8.onnx  ← production (faster)')
print('     compatibility_embedding.onnx       ← reference')